In [ ]:
import subprocess
from pathlib import Path
import geopandas as gpd
import tarfile
from osgeo import gdal
import shutil

boundary = gpd.read_file('data/vectors/greenland/GRL_adm0.shp').to_crs(epsg=3413)
ice_sheet = gpd.read_file('data/vectors/greenland/GRE_IceSheet_IMBIE2_v1.shp').to_crs(epsg=3413)
index = gpd.read_file('data/vectors/greenland/ArcticDEM_Mosaic_Index_v4_1_2m.shp').to_crs(epsg=3413)

# find area of Greenland surrounding the ice sheet
periphery = boundary.overlay(ice_sheet, how='difference')
periphery.to_file('data/vectors/greenland/periphery.shp')

# specify a folder in which to save downloaded ArcticDEM tiles
tar_folder = Path('data/rasters/greenland/tars')
tar_folder.mkdir(parents=True, exist_ok=True)

# function to check that downloaded files are complete and uncorrupted
# If any files corrupted, runs the download function again to ensure completeness of dataset
def check_tar_integrity(tar_path):
    try:
        with tarfile.open(tar_path, "r:*") as tar:
            tar.getmembers()  # Attempt to list members to verify integrity
        print(f"{tar_path.name}: OK")
    except EOFError as eof_error:
        print(f"{tar_path.name}: Corrupted (EOFError - {eof_error})")
        tar_path.unlink()  # Delete corrupted file
        print(f"{tar_path.name}: Deleted due to corruption.")
        get_data(tiles)
    except tarfile.TarError as tar_error:
        print(f"{tar_path.name}: Corrupted (TarError - {tar_error})")
        tar_path.unlink()  # Delete corrupted file
        print(f"{tar_path.name}: Deleted due to corruption.")
        get_data(tiles)

# Iterate through all .tar files in the directory and check their integrity
print('')
print('Checking downloaded tar files...')

for tar_file in tar_folder.rglob('*'):
    if tar_file.suffix in [".tar", ".gz"]:  # Check for .tar or .gz files
        print('')
        print(f"Checking {tar_file.name}...")
        check_tar_integrity(tar_file)
print('')
print('All tars downloaded successfully. Preparing to unpack dems...')
print('')

# create a new folder into which the dems will be unpacked
unpacked_folder = Path('data/rasters/greenland/unpacked')
unpacked_folder.mkdir(parents=True, exist_ok=True)

# create a new folder into which the dems will be deflated (necessary for whitebox operations)
dem_folder = Path('data/rasters/greenland/dems')
dem_folder.mkdir(parents=True, exist_ok=True)

periphery_shp = Path('data/vectors/greenland/periphery.shp')

# iterate through the tar files and unpack the dems into the previously specified folder
for tar_file in tar_folder.iterdir():
    with tarfile.open(tar_file, 'r') as tar:
        members = tar.getmembers()
        for member in members:
            if 'dem.tif' in member.name:
                unpacked = unpacked_folder / member.name
                if unpacked.exists():
                    print(f'{member.name} already unpacked. Skipping...')
                    print('')
                    continue
                else:
                    print(f'unpacking {member.name}...')
                    tar.extract(member, path=unpacked_folder)
                    options = ['COMPRESS=DEFLATE']
                    gdal.Warp(str(dem_folder / member.name), 
                                str(unpacked), 
                                cutlineDSName=str(periphery_shp), 
                                creationOptions=options)
                    print(f'{member.name} unpacked successfully')
                    print('')

print('')
print('Tidying up...')
print('')

# remove the interim folder
shutil.rmtree(unpacked_folder)
print('Greenland data ready for further analysis. \U0001F680')
print('')

/home/jamiemac/miniconda3/envs/geospatial/lib/python3.14/site-packages/geopandas/tools/overlay.py:358: UserWarning: `keep_geom_type=True` in overlay resulted in 10542 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  result = _collection_extract(result, geom_type, keep_geom_type_warning)



Checking downloaded tar files...

Checking 16_45_1_2_2m_v4.1.tar.gz...
16_45_1_2_2m_v4.1.tar.gz: OK

All tars downloaded successfully. Preparing to unpack dems...

unpacking 16_45_1_2_2m_v4.1_dem.tif...
16_45_1_2_2m_v4.1_dem.tif unpacked successfully


Tidying up...

Greenland data ready for further analysis. 🚀



In [6]:
periphery

,ID_0,ISO,NAME_ENGLI,NAME_ISO,NAME_FAO,NAME_LOCAL,NAME_OBSOL,NAME_VARIA,NAME_NONLA,NAME_FRENC,...,CARICOM,EU,CAN,ACP,Landlocked,AOSIS,SIDS,Islands,LDC,geometry
0,90,GRL,Greenland,GREENLAND,None,Kalaallit Nunaat,None,GrÃ¸nland|Greenland|Nunatta,None,Groenland,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,"MULTIPOLYGON (((-3296777.636 -2150907.907, -32..."
